In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import re
from datetime import datetime

def extract_dates(text):
    """
    Extract dates in various formats
    Formats: MM/DD/YYYY, DD-MM-YYYY, Month DD, YYYY, YYYY-MM-DD
    """
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',  # MM/DD/YYYY
        r'\d{1,2}-\d{1,2}-\d{4}',  # DD-MM-YYYY
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',  # Month DD, YYYY
        r'\d{4}-\d{2}-\d{2}'  # ISO format
    ]

    dates = []

    for pattern in patterns:
        matches = re.findall(pattern, text)
        dates.extend(matches)

    return dates


# Test
text = "Invoice date: 03/15/2024. Due: March 30, 2024"
print(extract_dates(text))

['03/15/2024', 'March 30, 2024']


In [2]:
import re

def extract_amounts(text):
    """
    Extract currency amounts
    Handles: $1,250.50, 1250.50, $1250
    """
    pattern = r'\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
    amounts = re.findall(pattern, text)

    # Convert to float
    cleaned = []
    for amount in amounts:
        # Remove $ and commas
        clean = amount.replace('$', '').replace(',', '')
        cleaned.append(float(clean))

    return cleaned


# Test
text = "Total: $1,250.50. Tax: $125.05. Subtotal: 1125.45"
print(extract_amounts(text))

[1250.5, 125.05, 112.0, 5.45]


In [5]:
import re

def extract_invoice_number(text):
    """
    Extract invoice/order numbers
    Patterns: INV-2024-001, #12345, ORDER-ABC123
    """
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'#\d{5,}',
        r'ORDER-[A-Z0-9]+',
        r'Invoice (?:Number|#):?\s*([A-Z0-9-]+)'
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            # Return full match or first captured group
            return match.group(1) if match.groups() else match.group(0)

    return None


# Test
text = "Invoice Number: INV-2024-001"
print(extract_invoice_number(text))

INV-2024-001


In [6]:
import spacy

# Load model (make sure it's installed in Kaggle first)
# !python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")

# Sample invoice text
text = """
Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith (john@acme.com)
Date: March 15, 2024
Amount Due: $1,250.50
"""

# Process text
doc = nlp(text)

# Extract entities
print("Found entities:")
for ent in doc.ents:
    print(f"{ent.text:20} {ent.label_:15} {spacy.explain(ent.label_)}")

Found entities:
Acme Corporation     ORG             Companies, agencies, institutions, etc.
123                  CARDINAL        Numerals that do not fall under another type
Main Street          FAC             Buildings, airports, highways, bridges, etc.
New York             GPE             Countries, cities, states
10001                DATE            Absolute or relative dates or periods
John Smith           PERSON          People, including fictional
March 15, 2024       DATE            Absolute or relative dates or periods
1,250.50             MONEY           Monetary values, including unit


In [7]:
def extract_entities(text):
    """
    Extract and organize entities by type
    """
    doc = nlp(text)

    entities = {
        'persons': [],
        'organizations': [],
        'locations': [],
        'dates': [],
        'money': []
    }

    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['persons'].append(ent.text)

        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)

        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text)

        elif ent.label_ == 'DATE':
            entities['dates'].append(ent.text)

        elif ent.label_ == 'MONEY':
            entities['money'].append(ent.text)

    return entities


# Test
result = extract_entities(text)

for entity_type, values in result.items():
    print(f"{entity_type}: {values}")
    

persons: ['John Smith']
organizations: ['Acme Corporation']
locations: ['New York']
dates: ['10001', 'March 15, 2024']
money: ['1,250.50']


In [10]:
from spacy import displacy

# Render in notebook
displacy.render(doc, style="ent", jupyter=True)

# Generate HTML string properly
html = displacy.render(doc, style="ent", page=True)

if html is not None:
    with open("entities.html", "w", encoding="utf-8") as f:
        f.write(html)
    print("Visualization saved to entities.html")
else:
    print("HTML generation failed (returned None)")

HTML generation failed (returned None)


In [ ]:
import json
import pytesseract
from PIL import Image

def process_invoice(image_path):
    """
    Complete pipeline: OCR → Extraction → JSON
    """

    # Step 1: OCR
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img)

    # Step 2: Regex extraction
    invoice_data = {
        'invoice_number': extract_invoice_number(text),
        'dates': extract_dates(text),
        'amounts': extract_amounts(text)
    }

    # Step 3: NER extraction
    entities = extract_entities(text)
    invoice_data.update(entities)

    # Step 4: Post-processing (safe checks)
    if invoice_data.get('amounts'):
        invoice_data['total_amount'] = max(invoice_data['amounts'])

    if invoice_data.get('dates'):
        invoice_data['invoice_date'] = invoice_data['dates'][0]

    return invoice_data


# Test (make sure file exists in Kaggle working directory)
result = process_invoice("")

print(json.dumps(result, indent=2))

In [11]:
import json
import pytesseract
from PIL import Image

def process_invoice(image_path):
    """
    Complete pipeline: OCR → Extraction → JSON
    """

    # Step 1: OCR
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img)

    # Step 2: Regex extraction
    invoice_data = {
        'invoice_number': extract_invoice_number(text),
        'dates': extract_dates(text),
        'amounts': extract_amounts(text)
    }

    # Step 3: NER extraction
    entities = extract_entities(text)
    invoice_data.update(entities)

    # Step 4: Post-processing (safe checks)
    if invoice_data.get('amounts'):
        invoice_data['total_amount'] = max(invoice_data['amounts'])

    if invoice_data.get('dates'):
        invoice_data['invoice_date'] = invoice_data['dates'][0]

    return invoice_data


# Test (make sure file exists in Kaggle working directory)
result = process_invoice("/kaggle/input/datasets/muneebafzaal6/ocrdataset23/images/13.jpg")

print(json.dumps(result, indent=2))

{
  "invoice_number": null,
  "dates": [
    "23506",
    "009049",
    "UE 10.06",
    "792986",
    "9796 5895 92387",
    "9333 1992"
  ],
  "amounts": [
    843.0,
    292.0,
    96.0,
    2.0,
    201.0,
    4.0,
    0.0,
    235.0,
    6.0,
    1.0,
    27.0,
    3.0,
    9.0,
    49.0,
    49.0,
    1.0,
    42.0,
    87.0,
    458.0,
    604.0,
    333.0,
    50.0,
    0.0,
    50.0,
    50.0,
    0.0,
    60.0,
    10.06,
    80.0,
    0.0,
    613.0,
    968.0,
    545.0,
    249.0,
    29.0,
    792.0,
    986.0,
    7.0,
    2.0,
    0.0,
    50.09,
    50.0,
    17.0,
    1.0,
    92.0,
    979.0,
    6.0,
    589.0,
    5.0,
    923.0,
    87.0,
    933.0,
    3.0,
    199.0,
    2.0,
    8.0,
    7.0
  ],
  "persons": [
    "MICHAEL",
    "GHARD"
  ],
  "organizations": [
    "TE",
    "REF douo7A2",
    "TANTATAAT"
  ],
  "locations": [],
  "money": [],
  "total_amount": 986.0,
  "invoice_date": "23506"
}


In [13]:
# Save to JSON file
output_file = "extracted_data.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2)

print(f"Results saved to {output_file}")

Results saved to extracted_data.json
